# Import Library

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd 
import numpy as np
from joblib import Parallel, delayed
import os

In [2]:
class Config:
    original = '1 Billion Citation Dataset, v1 (151).csv'
    extracted_df = '1 Billion Citation Dataset, v1 (150)(extracted).csv'
    original_text_done = '1 Billion Citation Dataset, v1 (150)(done_text).csv'

    checkpoint_path = 'checkpoint.csv'
    
    text_col_ori = "citationStringAnnotated"
    text_col = 'Title'

# Extract original data

In [4]:
original_df = pd.read_csv(Config.original)
original_df_copy = original_df.copy()
print(original_df.shape)
original_df.head(3)

(4516057, 5)


,doi,articleType,citationStyle,citationStringAnnotated,Unnamed: 4
0,10.1086/113099,3,0,"<author><family>Hummel</family>, <given>E.</gi...",NaN
1,10.1152/ajpendo.1989.257.6.e923,3,0,"<author><family>Vara</family>, <given>E.</give...",NaN
2,10.1016/j.compbiomed.2015.04.014,3,0,"<author><family>Rashed</family>, <given>Essam ...",NaN


In [20]:
import re
# Function to extract relevant fields from the text
def extract_fields(text):
    # Extract authors
    authors = re.findall(r"<author>(.*?)</author>", text)
    authors = ", ".join(re.sub(r"<.*?>", "", author) for author in authors) if authors else "no information"
    
    # Extract title
    title = re.search(r"<title>(.*?)</title>", text)
    title = title.group(1) if title else "no information"
    
    # Extract publisher
    publisher = re.search(r"<publisher>(.*?)</publisher>", text)
    publisher = publisher.group(1) if publisher else "no information"
    
    # Extract container-title (journal name)
    container_title = re.search(r"<container-title>(.*?)</container-title>", text)
    container_title = container_title.group(1) if container_title else "no information"
    
    # Extract year
    year = re.search(r"<issued><year>(\d{4})</year></issued>", text)
    year = year.group(1) if year else "no information"
    
    # Extract volume
    volume = re.search(r"<volume>(.*?)</volume>", text)
    volume = volume.group(1) if volume else "no information"
    
    # Extract issue
    issue = re.search(r"<issue>(.*?)</issue>", text)
    issue = issue.group(1) if issue else "no information"
    
    # Extract pages
    pages = re.search(r"<page>(.*?)</page>", text)
    pages = pages.group(1) if pages else "no information"
    
    # Extract URL
    url = re.search(r"<URL>(.*?)</URL>", text)
    url = url.group(1) if url else "no information"
    
    return {
        "Authors": authors,
        "Title": title,
        "Publisher": publisher,
        "Journal": container_title,
        "Year": year,
        "Volume": volume,
        "Issue": issue,
        "Pages": pages,
        "URL": url
    }


# Extract file
extracted_data = original_df[Config.text_col_ori].apply(extract_fields)
extracted_data = pd.DataFrame(extracted_data.tolist())
print(extracted_data.shape)
extracted_data.head()

(4538038, 9)


,Authors,Title,Publisher,Journal,Year,Volume,Issue,Pages,URL
0,no information,“Front Matter.”,no information,Modern Philology,1991,88,4,i–ii,http://dx.doi.org/10.1086/mp.88.4.438262
1,"Tao Hao, Luo Zhigang and Liu Jinde",“An end-to-end scheduling approach for real-ti...,IEEE,"2002 IEEE Region 10 Conference on Computers, C...",no information,no information,no information,no information,http://dx.doi.org/10.1109/tencon.2002.1181278
2,"Collet, Pierre, Martinez, Servet and Martin, J...",“Asymptotic Laws for One-Dimensional Diffusion...,no information,The Annals of Probability,1995,23,3,1300–1314,http://dx.doi.org/10.1214/aop/1176988185
3,"Thuler, Luiz Claudio Santos, de Menezes, Raque...",“Cancer cases attributable to alcohol consumpt...,no information,Alcohol,2016,54,no information,23–26,http://dx.doi.org/10.1016/j.alcohol.2016.05.004
4,"Belle, Fabiën N., Kasteler, Rahel, Schindera, ...",“No evidence of overweight in long-term surviv...,no information,Cancer,2018,124,17,3576–3585,http://dx.doi.org/10.1002/cncr.31599


In [21]:
# Drop URL
extracted_data.drop(columns='URL', inplace=True)

# Litle clean before export
# Title column
# Removal of punctuations
import string
PUNCT = string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', PUNCT + '“”ʼ»«’–‐×‘'))
extracted_data[Config.text_col] = extracted_data[Config.text_col].astype(str).apply(remove_punctuation)
# Year column
extracted_data['Year'] = extracted_data['Year'].apply(lambda x: int(x) if str(x).isdigit() else None)
# Issue column
extracted_data['Issue'] = extracted_data['Issue'].apply(lambda x: int(x) if str(x).isdigit() else None)
# Pages
extracted_data['Pages'] = extracted_data['Pages'].apply(lambda x: x if str(x).isdigit() or re.match(r'^\d+–\d+$', str(x)) else "no information")
# Volume column
extracted_data['Volume'] = extracted_data['Volume'].apply(lambda x: int(x) if str(x).isdigit() else None)

# Remove URLS from text column
def remove_urls(text):
    # Check if the input is a Series
    if isinstance(text, pd.Series):
        return text.str.replace(r'https?://\S+|www\.\S+', 'no information', regex=True)
    else:
        url_pattern = re.compile(r'https?://\S+|www\.\S+')
        return url_pattern.sub('no information', text)

text_columns = ['Authors', 'Publisher', 'Journal']
extracted_data[text_columns] = extracted_data[text_columns].astype(str).apply(remove_urls)

# Export extracted file
extracted_data.to_csv(Config.extracted_df, index=False)
print(extracted_data.shape)
extracted_data.head()

(4538038, 8)


,Authors,Title,Publisher,Journal,Year,Volume,Issue,Pages
0,no information,Front Matter,no information,Modern Philology,1991.0,88.0,4.0,no information
1,"Tao Hao, Luo Zhigang and Liu Jinde",An endtoend scheduling approach for realtime C...,IEEE,"2002 IEEE Region 10 Conference on Computers, C...",NaN,NaN,NaN,no information
2,"Collet, Pierre, Martinez, Servet and Martin, J...",Asymptotic Laws for OneDimensional Diffusions ...,no information,The Annals of Probability,1995.0,23.0,3.0,1300–1314
3,"Thuler, Luiz Claudio Santos, de Menezes, Raque...",Cancer cases attributable to alcohol consumpti...,no information,Alcohol,2016.0,54.0,NaN,23–26
4,"Belle, Fabiën N., Kasteler, Rahel, Schindera, ...",No evidence of overweight in longterm survivor...,no information,Cancer,2018.0,124.0,17.0,3576–3585


# Text processing with extracted data

In [3]:
# Import extracted data
extracted_data = pd.read_csv(Config.extracted_df)

In [ ]:
extracted_data

In [23]:
# List "no information" indexs in Title columns
title_no_inf_list = extracted_data.index[extracted_data[Config.text_col] == "no information"].tolist()
print(f"total index will be droped: [{len(title_no_inf_list)}]")

# Drop those index
extracted_df_1 = extracted_data.drop(title_no_inf_list).reset_index(drop=True)
print(extracted_df_1.shape)
extracted_df_1.head()

total index will be droped: [237658]
(4300380, 8)


,Authors,Title,Publisher,Journal,Year,Volume,Issue,Pages
0,no information,Front Matter,no information,Modern Philology,1991.0,88.0,4.0,no information
1,"Tao Hao, Luo Zhigang and Liu Jinde",An endtoend scheduling approach for realtime C...,IEEE,"2002 IEEE Region 10 Conference on Computers, C...",NaN,NaN,NaN,no information
2,"Collet, Pierre, Martinez, Servet and Martin, J...",Asymptotic Laws for OneDimensional Diffusions ...,no information,The Annals of Probability,1995.0,23.0,3.0,1300–1314
3,"Thuler, Luiz Claudio Santos, de Menezes, Raque...",Cancer cases attributable to alcohol consumpti...,no information,Alcohol,2016.0,54.0,NaN,23–26
4,"Belle, Fabiën N., Kasteler, Rahel, Schindera, ...",No evidence of overweight in longterm survivor...,no information,Cancer,2018.0,124.0,17.0,3576–3585


## Group unique title

In [24]:
unique_df = extracted_df_1.groupby(Config.text_col).apply(lambda x: x.index.tolist()).reset_index(name='index')
unique_df.columns = [Config.text_col, 'index']
print(unique_df.shape)
unique_df.head()

(4644, 2)


,Title,index
0,\t\t\t\tFLOW STRUCTURE AND TURBULENCE MODIFICA...,"[565, 3420, 6275, 9130, 11985, 14840, 17695, 2..."
1,\t\t\t\tFLOW STRUCTURE AND TURBULENCE MODIFICA...,"[72103, 109255, 123530, 126385, 384205, 500574..."
2,Xcat a Novel Mouse Model for NanceHo...,"[71761, 108913, 123188, 126043, 358168, 383863..."
3,Xcat a novel mouse model for NanceHo...,"[223, 3078, 5933, 8788, 11643, 14498, 17353, 2..."
4,Nkx25 Transactivates the EtsRelated Protei...,"[73744, 110896, 125171, 128026, 360151, 385846..."


In [25]:
total_idx = 0
for idx, col in unique_df.iterrows():
    total_idx += len(col['index'])

total_idx == extracted_df_1.shape[0]

True

## Process title

In [26]:
pd.options.mode.chained_assignment = None
df = unique_df[[Config.text_col]]
df[[Config.text_col]] = df[[Config.text_col]].astype(str)

# Removal of punctuations
import string
PUNCT = string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', PUNCT + '“”ʼ»«’–‐×‘'))


# Removal of stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))
STOPW = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPW])
    

# Removal of frequent words
from collections import Counter
cnt = Counter()
for x in df.columns:
    for text in df[x].values:
        for word in text.split():
            cnt[word] += 1
cnt.most_common(10)
FREQWS = set([w for (w, wc) in cnt.most_common(10)])
def remove_freqwords(text):
    return " ".join([word for word in str(text).split() if word not in FREQWS])

# Stemming
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])
    

# Lemmatization
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
    

# Removal of URLs
import re
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

# Removal of HTML Tags
    # Solution 1
def remove_html1(text):
    html_pattern = re.compile('<.*?>')
    return html_pattern.sub(r'', text)
    # Solution 2
from bs4 import BeautifulSoup
def remove_html2(text):
    return BeautifulSoup(text, "lxml").text

# Spelling Correction
from spellchecker import SpellChecker
spell = SpellChecker()
def correct_spellings(text):
    corrected_text = []
    misspelled_words = spell.unknown(text.split())
    for word in text.split():
        if word in misspelled_words:
            correction = spell.correction(word)
            # If no correction found, keep the original word
            corrected_text.append(correction if correction else word)
        else:
            corrected_text.append(word)
    return " ".join(corrected_text)


def process(text, is_lower=False,
            rev_punctuation=False, rev_stopwords=False,
            stemming=False, rev_urls=False, rev_html1=False,
            lemmatizing=False, rev_frequent=False, spelling_correcting=False):
    # lowercase
    if is_lower:
        text = text.lower()
    # remove html
    if rev_html1:
        text = remove_html1(text)
    # remove urls
    if rev_urls:
        text = remove_urls(text)
    # remove punctuation
    if rev_punctuation:
        text = remove_punctuation(text)
    # remove stopwords
    if rev_stopwords:
        text = remove_stopwords(text)
    # remove frequent words
    if rev_frequent:
        text = remove_freqwords(text)
    # lemmatize words (return to basic form of the word)
    if lemmatizing:
        text = lemmatize_words(text)
    # correct spelling 
    if spelling_correcting:
        text = correct_spellings(text)
    # stemming
    if stemming:
        text = stem_words(text)
    return text

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nguyentrunganhonichan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/nguyentrunganhonichan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
total_na_idx = 0
# Convert STOPW to a set for faster lookup
STOPW_set = set(STOPW)

# Apply .lower() to the entire 'Title' Series once
lower_titles = unique_df['Title'].str.lower()

# Check for stop words in titles using vectorized operations
total_na_idx = sum(1 for title in lower_titles if title in STOPW_set)  # {{ edit_1 }}

print(f"{total_na_idx} NaN in Title col")

0 NaN in Title col


In [28]:
df[Config.text_col] = df[Config.text_col].apply(lambda text: process(text, is_lower=True,
                                                                    rev_punctuation=True, rev_stopwords=True,
                                                                    lemmatizing=True, rev_frequent=True))

In [29]:
# Checkpoint file path
checkpoint_file = Config.checkpoint_path

# Load checkpoint if it exists
if os.path.exists(checkpoint_file):
    checkpoint_df = pd.read_csv(checkpoint_file)
    start_index = len(checkpoint_df)

    print(start_index)
    print(f"main file:  {unique_df[Config.text_col].iloc[start_index-1]}")
    print(f"checkpoint: {checkpoint_df[Config.text_col].iloc[start_index-1]} \n")
else:
    print("NO Checkpoint exist!!!")
    start_index = 0
    print(start_index)

# Process in batches and save checkpoints
batch_size = 100  # Adjust batch size as needed
for i in range(start_index, len(df), batch_size):
    batch = df.iloc[i:i + batch_size]
    print(f"start at index: {i}\n {df[Config.text_col].iloc[i]}")
    batch[Config.text_col] = Parallel(n_jobs=-1)(
                                                delayed(process)(text, spelling_correcting=True, stemming=True)
                                                for text in batch[Config.text_col]
)
    batch[Config.text_col] = Parallel(n_jobs=-1)(
                                                delayed(process)(text, stemming=True)
                                                for text in batch[Config.text_col]
)
    # Save the processed batch to a checkpoint
    batch.to_csv(checkpoint_file, mode='a', header=not os.path.exists(checkpoint_file), index=False)
    print(f"finished {i+batch_size}")

NO Checkpoint exist!!!
0
start at index: 0
 flow structure turbulence modification downward bubbly flow
finished 100
start at index: 100
 018x03bcm cmos 9mw currentmode flf linear phase filter gain boost
finished 200
start at index: 200
 review solar drying technology
finished 300
start at index: 300
 acknowledgment 19
finished 400
start at index: 400
 advance systemic treatment neuroendocrine tumor era molecular therapy
finished 500
start at index: 500
 annual variation common eider egg size effect temperature clutch size laying date laying sequence
finished 600
start at index: 600
 automatic symbolic evaluation nonideal effect si circuit behavior
finished 700
start at index: 700
 biodiversity odonata rice pattukkottai tamil nadu
finished 800
start at index: 800
 condenser
finished 900
start at index: 900
 characteristic new absorbent system heteropoly compound solution hltsubrt2ltsubrts removal
finished 1000
start at index: 1000
 cold atmospheric plasma manipulation protein food syst

In [30]:
# Import back checkpoint 
checkpoint = pd.read_csv(Config.checkpoint_path)

# Replace unique_df with processed Title
unique_df['Title'] = checkpoint['Title']

# Repace new Title to extrected file
for row, col in unique_df.iterrows():
    extracted_df_1[Config.text_col].iloc[col['index']] = col['Title']

extracted_df_1.isnull().sum()

Authors            0
Title              0
Publisher          0
Journal            0
Year          774415
Volume       1334639
Issue        2927723
Pages              0
dtype: int64

# Concat with original file and EXPORT

In [31]:
original_df_copy = original_df_copy.drop(title_no_inf_list).reset_index(drop=True)

# double check
for x in range(5):
    ind = np.random.randint(low=0, high=original_df_copy.shape[0])
    print(f"oriindex:{ind}:  {original_df_copy[Config.text_col_ori].iloc[ind]}")
    print(f"index:{ind}:  {extracted_df_1[Config.text_col].iloc[ind]}")


# Một số hàng có chỉ có stopword nên bị xoá hết. --> đầu ra có thể có NaN
# Một số hàng chỉ có số và đó là lỗi của data.
# Journal có url

oriindex:1309200:  <author><family>Schäfers</family>, <given>M.</given> & <family>Tölle</family>, <given>T.R.</given></author>. <issued><year>2013</year></issued>. <title>Aktuelle Therapie neuropathischer Schmerzen</title>. <container-title>Der Nervenarzt</container-title>, <volume>84</volume>(<issue>12</issue>): <page>1445–1450</page>. https://doi.org/<DOI>10.1007/s00115-012-3623-5</DOI>
index:1309200:  aktuel therapi neuropathisch schmerzen
oriindex:424097:  <author><family>Aschmann</family>, <given>H.</given></author>. (<issued><year>1976</year></issued>) <title>Latin American development: A geographical perspective</title>. <container-title>Journal of Historical Geography</container-title>, <publisher>Elsevier BV</publisher> <volume>2</volume>, <page>379–380</page>.
index:424097:  satin america develop geograph perspect
oriindex:3597255:  <author><family>Kim</family>, <given>D. H.</given> / <family>Murovic</family>, <given>J. A.</given> / <family>Kim</family>, <given>Y.-Y.</given> 

In [32]:
original_df_copy = original_df_copy.drop(columns=[Config.text_col_ori, 'Unnamed: 4'])
original_text_done = pd.concat([original_df_copy,extracted_df_1], axis='columns')
original_text_done.to_csv(Config.original_text_done, index=False)